In [ ]:
import sys
sys.path.append('..')

from tools.geometry import generate_detector
from tools.simulation import setup_event_simulator
import jax
import jax.numpy as jnp
import time
import math
from jax import jit


from matplotlib import pyplot as plt
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10

from tools.geometry import Cylinder, Sphere, Box

In [ ]:
import numpy as np
import math
import plotly.graph_objects as go
import jax
import jax.numpy as jnp
import json

import sys
sys.path.append('..')

from tools.geometry import Cylinder, Sphere, Box


def get_detector_bounds(detector):
    """Extract detector bounds based on detector type."""
    detector_type = detector.__class__.__name__
    
    if 'Cylinder' in detector_type:
        return {
            'type': 'cylinder',
            'r': detector.r,
            'H': detector.H
        }
    elif 'Sphere' in detector_type:
        return {
            'type': 'sphere',
            'r': detector.r
        }
    elif 'Box' in detector_type:
        return {
            'type': 'box',
            'x': detector.L,
            'y': detector.W,
            'z': detector.H
        }
    else:
        raise ValueError(f"Unknown detector type: {detector_type}")


def generate_random_event_params(key, detector_bounds, fraction=1.0):
    """Generate random event parameters based on detector geometry."""
    if detector_bounds['type'] == 'cylinder':
        r_vert = jax.random.uniform(key, shape=(), minval=0, maxval=detector_bounds['r'] * fraction)
        key, _ = jax.random.split(key)
        theta = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
        key, _ = jax.random.split(key)
        z_vert = jax.random.uniform(key, shape=(), minval=-detector_bounds['H']/2 * fraction, 
                                   maxval=detector_bounds['H']/2 * fraction)
        position = jnp.array([r_vert * jnp.cos(theta), r_vert * jnp.sin(theta), z_vert])
        
    elif detector_bounds['type'] == 'sphere':
        u = jax.random.uniform(key, shape=())
        key, _ = jax.random.split(key)
        cos_theta = jax.random.uniform(key, shape=(), minval=-1, maxval=1)
        key, _ = jax.random.split(key)
        phi = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
        
        r = detector_bounds['r'] * fraction * jnp.cbrt(u)
        sin_theta = jnp.sqrt(1 - cos_theta**2)
        position = jnp.array([r * sin_theta * jnp.cos(phi), 
                             r * sin_theta * jnp.sin(phi), 
                             r * cos_theta])
        
    elif detector_bounds['type'] == 'box':
        position = jax.random.uniform(key, shape=(3,), 
                                    minval=jnp.array([-detector_bounds['x']/2, 
                                                     -detector_bounds['y']/2, 
                                                     -detector_bounds['z']/2]) * fraction,
                                    maxval=jnp.array([detector_bounds['x']/2, 
                                                     detector_bounds['y']/2, 
                                                     detector_bounds['z']/2]) * fraction)

    # Random direction
    key, _ = jax.random.split(key)
    phi = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
    key, _ = jax.random.split(key)
    cos_theta = jax.random.uniform(key, shape=(), minval=-1, maxval=1)
    sin_theta = jnp.sqrt(1 - cos_theta**2)
    direction = jnp.array([sin_theta * jnp.cos(phi), sin_theta * jnp.sin(phi), cos_theta])
    
    # Random energy
    key, _ = jax.random.split(key)
    energy = jax.random.uniform(key, shape=(), minval=250., maxval=1750.)
    
    return position, direction, energy


def is_point_inside_detector(point, detector_bounds, fraction=1.0):
    """Check if a point is inside the detector bounds with given fraction."""
    x, y, z = point
    
    if detector_bounds['type'] == 'cylinder':
        r_max = detector_bounds['r'] * fraction
        h_max = detector_bounds['H'] * fraction / 2
        r_point = np.sqrt(x**2 + y**2)
        return r_point <= r_max and abs(z) <= h_max
        
    elif detector_bounds['type'] == 'sphere':
        r_max = detector_bounds['r'] * fraction
        r_point = np.sqrt(x**2 + y**2 + z**2)
        return r_point <= r_max
        
    elif detector_bounds['type'] == 'box':
        hx = detector_bounds['x'] * fraction / 2
        hy = detector_bounds['y'] * fraction / 2
        hz = detector_bounds['z'] * fraction / 2
        return abs(x) <= hx and abs(y) <= hy and abs(z) <= hz
        
    return False


def generate_grid_points_local(center, local_size, L, detector_bounds, fraction=1.0):
    """
    Generate 3D cubic grid points within a local region, filtered by detector bounds.
    
    Parameters:
    -----------
    center : tuple
        Center of local search region (x, y, z)
    local_size : float
        Half-size of local cubic search region
    L : float
        Grid spacing
    detector_bounds : dict
        Detector geometry bounds
    fraction : float
        Fraction of detector dimensions to use (0 < fraction <= 1)
    """
    cx, cy, cz = center

    # cubic grid spacing
    dx = dy = dz = L

    # bounds of local search region
    x_min, x_max = cx - local_size, cx + local_size
    y_min, y_max = cy - local_size, cy + local_size
    z_min, z_max = cz - local_size, cz + local_size

    xs = np.arange(x_min, x_max + dx, dx)
    ys = np.arange(y_min, y_max + dy, dy)
    zs = np.arange(z_min, z_max + dz, dz)

    pts = []
    for x in xs:
        for y in ys:
            for z in zs:
                if is_point_inside_detector((x, y, z), detector_bounds, fraction):
                    pts.append((x, y, z))

    # print('exp: ', len(xs)**3)
    # print('obs: ', len(pts))

    return np.array(pts)


def hierarchical_position_search(target, detector_bounds, n_div=5, levels=6, fraction=1.0, min_L=0.01):
    """
    Unified hierarchical position search for any detector geometry.
    Uses a regular 3D cubic grid and filters points based on detector bounds.
    
    Parameters:
    -----------
    target : array-like
        Target position (x, y, z)
    detector_bounds : dict
        Detector geometry bounds from get_detector_bounds()
    n_div : int
        Number of divisions in the grid
    levels : int
        Number of refinement levels
    fraction : float
        Fraction of the detector where we look for the vertex position
    min_L : float
        Minimum grid spacing
    """
    center = (0.0, 0.0, 0.0)  # start at detector center
    
    # Initial local search size based on detector type
    if detector_bounds['type'] == 'cylinder':
        local_size = max(detector_bounds['r'], detector_bounds['H']/2) * fraction
    elif detector_bounds['type'] == 'sphere':
        local_size = detector_bounds['r'] * fraction
    elif detector_bounds['type'] == 'box':
        local_size = max(detector_bounds['x'], detector_bounds['y'], detector_bounds['z']) * fraction / 2

    path = []

    for lvl in range(levels):
        L = (2*local_size)/n_div
        if L < min_L:
            break

        pts = generate_grid_points_local(center, local_size, L, detector_bounds, fraction)
        if pts.size == 0:
            break

        dists = np.linalg.norm(pts - np.array(target)[None, :], axis=1)
        best_idx = int(np.argmin(dists))
        best_pt = pts[best_idx]
        best_dist = float(dists[best_idx])

        path.append({
            "level": lvl,
            "L": L,
            "num_points": len(pts),
            "best_point": best_pt,
            "best_dist": best_dist,
            "center": center,
            "local_size": local_size
        })

        center = (best_pt[0], best_pt[1], best_pt[2])
        local_size = L/2
        
    return path

In [ ]:
# Load detector configurations and create detector instances
def load_detector_from_config(config_path):
    """Load detector from configuration file."""
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    detector_type = config['detector_type']
    geom = config['geometry_definitions']
    
    if detector_type == 'cylinder':
        return Cylinder(
            radius=geom['radius'],
            height=geom['height'],
            n_sensors=geom['n_sensors'],
            sensor_radius=geom['sensor_radius']
        )
    elif detector_type == 'sphere':
        return Sphere(
            radius=geom['radius'],
            n_sensors=geom['n_sensors'],
            sensor_radius=geom['sensor_radius']
        )
    elif detector_type == 'box':
        return Box(
            length=geom['length'],
            width=geom['width'],
            height=geom['height'],
            n_sensors=geom['n_sensors'],
            sensor_radius=geom['sensor_radius']
        )
    else:
        raise ValueError(f"Unknown detector type: {detector_type}")


# Test detectors
detectors = {
    'HK': load_detector_from_config('../config/HK_geom_config.json'),
    'JUNO': load_detector_from_config('../config/JUNO_geom_config.json'),
    'MidBox': load_detector_from_config('../config/MidBox_geom_config.json')
}

print("Loaded detectors:")
for name, detector in detectors.items():
    bounds = get_detector_bounds(detector)
    print(f"{name}: {detector.__class__.__name__}, bounds: {bounds}")

In [ ]:
# Test unified grid search with random targets for each detector
key = jax.random.PRNGKey(42)
fraction = 1.0

print("=" * 80)
print("UNIFIED GRID SEARCH VALIDATION")
print("=" * 80)

for detector_name, detector in detectors.items():
    print(f"\n{'-'*50}")
    print(f"TESTING {detector_name} ({detector.__class__.__name__})")
    print(f"{'-'*50}")
    
    detector_bounds = get_detector_bounds(detector)
    
    # Generate random target position within detector
    key, subkey = jax.random.split(key)
    target_pos, target_dir, target_energy = generate_random_event_params(
        subkey, detector_bounds, fraction=fraction
    )
    
    print(f"Target position: {target_pos}")
    print(f"Detector bounds: {detector_bounds}")
    print(f"Search fraction: {fraction}")
    
    # Perform hierarchical search
    path = hierarchical_position_search(
        target=target_pos, 
        detector_bounds=detector_bounds, 
        n_div = 2,
        fraction = fraction,
        levels=8
    )
    
    print(f"\nSearch Results:")
    for step in path:
        lvl = step["level"]
        print(f"  Level {lvl}: L={step['L']:.4f}, num_pts={step['num_points']}, "
              f"best_pt={step['best_point']}, dist={step['best_dist']:.4f}")
    
    if path:
        final_error = path[-1]['best_dist']
        print(f"\nFinal error: {final_error:.6f}")
        print(f"Target found within {final_error:.6f} units")

In [ ]:
# Visualization of search results
def visualize_search_results(detector_name, detector, target_pos, path, detector_bounds, fraction):
    """Visualize the hierarchical search results."""
    fig = go.Figure()
    colors = ["red", "orange", "green", "blue", "purple", "black"]
    
    # Plot search points for each level
    for i, step in enumerate(path):
        pts = generate_grid_points_local(
            step["center"], step["local_size"], step["L"], detector_bounds, fraction
        )
        if len(pts) > 0:
            fig.add_trace(go.Scatter3d(
                x=pts[:,0], y=pts[:,1], z=pts[:,2],
                mode="markers",
                marker=dict(size=8, color=colors[i % len(colors)], opacity=0.6),
                name=f"Level {i} (L={step['L']:.3f})"
            ))
            
            # Mark best point for this level
            bp = step["best_point"]
            fig.add_trace(go.Scatter3d(
                x=[bp[0]], y=[bp[1]], z=[bp[2]],
                mode="markers",
                marker=dict(size=8, color=colors[i % len(colors)], symbol="x"),
                name=f"Best L{i}"
            ))
    
    # Mark target position
    fig.add_trace(go.Scatter3d(
        x=[target_pos[0]], y=[target_pos[1]], z=[target_pos[2]],
        mode="markers",
        marker=dict(size=12, color="black", symbol="diamond"),
        name="Target"
    ))

    margin = 1.1
    # Set appropriate axis ranges based on detector type
    if detector_bounds['type'] == 'cylinder':
        r_max = detector_bounds['r'] * fraction * margin
        h_max = detector_bounds['H'] * fraction / 2 * margin
        fig.update_layout(
            scene=dict(
                xaxis=dict(range=[-r_max, r_max]),
                yaxis=dict(range=[-r_max, r_max]),
                zaxis=dict(range=[-h_max, h_max]),
            )
        )
    elif detector_bounds['type'] == 'sphere':
        r_max = detector_bounds['r'] * fraction * margin
        fig.update_layout(
            scene=dict(
                xaxis=dict(range=[-r_max, r_max]),
                yaxis=dict(range=[-r_max, r_max]),
                zaxis=dict(range=[-r_max, r_max]),
            )
        )
    elif detector_bounds['type'] == 'box':
        hx = detector_bounds['x'] * fraction / 2 * margin
        hy = detector_bounds['y'] * fraction / 2 * margin
        hz = detector_bounds['z'] * fraction / 2 * margin
        fig.update_layout(
            scene=dict(
                xaxis=dict(range=[-hx, hx]),
                yaxis=dict(range=[-hy, hy]),
                zaxis=dict(range=[-hz, hz]),
            )
        )
    
    fig.update_layout(
        title=f"Unified Grid Search - {detector_name} ({detector.__class__.__name__})",
        showlegend=True,
        scene=dict(aspectmode='data')
    )
    
    return fig

# Generate visualizations for each detector
print("\n" + "=" * 80)
print("GENERATING VISUALIZATIONS")
print("=" * 80)

fraction = 1.0
# Re-run searches and visualize
key = jax.random.PRNGKey(42)
for detector_name, detector in detectors.items():
    detector_bounds = get_detector_bounds(detector)
    
    # Generate same random target as before
    key, subkey = jax.random.split(key)
    target_pos, _, _ = generate_random_event_params(subkey, detector_bounds, fraction=fraction)
    print(target_pos)
    # Perform search
    path = hierarchical_position_search(
        target=target_pos, 
        detector_bounds=detector_bounds, 
        n_div = 2,
        fraction = 1.0,
        levels=7
    )
    
    # Create visualization
    fig = visualize_search_results(detector_name, detector, target_pos, path, detector_bounds, fraction)
    fig.show()
    
    print(f"Displayed visualization for {detector_name}")
    break